# Compute VAEP values and top players (Impect)

In [ ]:
%load_ext autoreload
%autoreload 2
import json
import warnings
from pathlib import Path
import pandas as pd
import tqdm
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

PROJECT_ROOT = Path('..').resolve()
CONFIG_CANDIDATES = [
    PROJECT_ROOT / 'private/impect-pipeline/config/my_iteration.json',
    PROJECT_ROOT / 'docs/documentation/data/impect_open_data.example.json',
]
CONFIG_PATH = next((p for p in CONFIG_CANDIDATES if p.is_file()), CONFIG_CANDIDATES[-1])
with open(CONFIG_PATH) as f:
    CFG = json.load(f)
OUT = PROJECT_ROOT / 'data/impect' / CFG['output_basename']
spadl_h5 = OUT / 'spadl-impect.h5'
features_h5 = OUT / 'features.h5'
labels_h5 = OUT / 'labels.h5'
predictions_h5 = OUT / 'predictions.h5'
print('config:', CONFIG_PATH)
print('output:', OUT)

def games_with_actions(path):
    with pd.HDFStore(path) as store:
        games = store['games']
        ids = {int(k.rsplit('_', 1)[-1]) for k in store.keys() if k.startswith('/actions/game_')}
    return games[games.game_id.isin(ids)].sort_values('game_date').reset_index(drop=True)

import socceraction.spadl as spadl
import socceraction.vaep.formula as vaepformula

In [ ]:
with pd.HDFStore(spadl_h5) as s:
    games = games_with_actions(spadl_h5)
    players = s['players'] if 'players' in s else pd.DataFrame()
    teams = s['teams'] if 'teams' in s else pd.DataFrame()
    pg = s['player_games'] if 'player_games' in s else pd.DataFrame()
print(len(games), 'games')


## VAEP values

In [ ]:
A = []
for game in tqdm.tqdm(list(games.itertuples()), desc='rate'):
    actions = pd.read_hdf(spadl_h5, f'actions/game_{game.game_id}')
    actions = spadl.add_names(actions)
    if len(players):
        actions = actions.merge(players, on='player_id', how='left')
    if len(teams):
        actions = actions.merge(teams, on='team_id', how='left')
    preds = pd.read_hdf(predictions_h5, f'game_{game.game_id}')
    values = vaepformula.value(actions, preds.scores, preds.concedes)
    A.append(pd.concat([actions, preds, values], axis=1))
A = pd.concat(A).reset_index(drop=True)


## Top players (per 90, min 180 minutes)

In [ ]:
playersR = A.groupby('player_id').agg(
    vaep_value=('vaep_value', 'sum'),
    offensive_value=('offensive_value', 'sum'),
    defensive_value=('defensive_value', 'sum'),
    count=('vaep_value', 'count'),
).reset_index()
if len(players):
    playersR = playersR.merge(players, on='player_id', how='left')
if len(pg):
    mp = pg[pg.game_id.isin(games.game_id)].groupby('player_id').minutes_played.sum().reset_index()
    playersR = playersR.merge(mp, on='player_id', how='left')
else:
    playersR['minutes_played'] = playersR['count']
playersR = playersR[playersR.minutes_played > 180]
playersR['vaep_rating'] = playersR.vaep_value * 90 / playersR.minutes_played
playersR['offensive_rating'] = playersR.offensive_value * 90 / playersR.minutes_played
playersR['defensive_rating'] = playersR.defensive_value * 90 / playersR.minutes_played
playersR.sort_values('vaep_rating', ascending=False).head(15)
